# 🧪 AIGI Detector Performance Comparison Benchmark
이 노트북은 **OmniAID**, **SIDA**, **BR-Gen** 세 모델의 성능을 비교 분석합니다.

### 측정 지표:
1. Accuracy, 2. Precision, 3. Recall (핵심), 4. F1 Score, 5. FPR, 6. Inference Time

In [1]:
import os
import time
import torch
import torch.nn as nn
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
from torchvision import transforms
from huggingface_hub import hf_hub_download
import numpy as np
import sys

# 각 모델 경로 추가
sys.path.append('./OmniAID')
sys.path.append('./SIDA')
sys.path.append('./BR-Gen')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Current Device: {device}")

Current Device: cpu


c:\Users\june4\miniconda3\envs\LP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 모델 로드 (Model Initialization)
각 모델의 아키텍처와 가중치를 불러옵니다.

In [2]:
def load_omniaid():
    from models.OmniAID import OmniAID
    from types import SimpleNamespace
    import json
    
    with open('./OmniAID/config.json', 'r') as f:
        config = SimpleNamespace(**json.load(f))
    
    model = OmniAID(config=config)
    # 가중치 파일이 없을 경우 다운로드 로직 (OmniAID 예시)
    ckpt_path = './OmniAID/checkpoint_omniaid_mirage.pth'
    if not os.path.exists(ckpt_path):
        ckpt_path = hf_hub_download(repo_id="Yunncheng/OmniAID", filename="checkpoint_omniaid_mirage.pth", local_dir="./OmniAID")
    
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)
    model.to(device).eval()
    return model

def load_sida():
    # SIDA 저장소 구조에 맞게 모델 임포트 (예시)
    # 실제 SIDA 모델 클래스 이름과 가중치 경로를 확인 후 수정 필요
    print("Loading SIDA Model...")
    return None # Placeholder

def load_brgen():
    # BR-Gen 저장소 구조에 맞게 모델 임포트 (예시)
    print("Loading BR-Gen Model...")
    return None # Placeholder

# 벤치마크 실행을 위해 OmniAID만 우선 활성화 (다른 모델은 가중치/코드 연결 필요)
model_omniaid = load_omniaid()

Initializing in HYBRID MoE architectural mode.


Loading weights: 100%|██████████| 590/590 [00:00<00:00, 19973.36it/s]


Rank allocation: total_rank=1024, r_main=1016, num_experts=6, rank_per_expert=8


## 2. 벤치마크 엔진 (Benchmark Engine)
지표 계산 및 성능 평가를 수행합니다.

In [3]:
def run_benchmark(model, model_name, test_data_path):
    """
    test_data_path 구조:
    - /real/*.jpg
    - /fake/*.jpg
    """
    tp, fp, fn, tn = 0, 0, 0, 0
    inference_times = []
    
    # 전처리 (OmniAID 기준)
    preprocess = transforms.Compose([
        transforms.Resize((336, 336)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.481, 0.457, 0.408], std=[0.268, 0.261, 0.275]),
    ])

    # 평가 루프
    for label_type in ['real', 'fake']:
        folder = os.path.join(test_data_path, label_type)
        if not os.path.exists(folder): continue
        
        for img_name in os.listdir(folder):
            img_path = os.path.join(folder, img_name)
            img = Image.open(img_path).convert('RGB')
            img_tensor = preprocess(img).unsqueeze(0).to(device)
            
            # 시간 측정 시작
            start_time = time.time()
            with torch.no_grad():
                outputs = model(img_tensor)
                # OmniAID 결과: {'prob': prob_tensor}
                prob = float(outputs['prob'][0]) 
                pred = 1 if prob > 0.5 else 0
            end_time = time.time()
            inference_times.append(end_time - start_time)
            
            # Confusion Matrix 계산
            if label_type == 'fake': # Positive 클래스
                if pred == 1: tp += 1
                else: fn += 1
            else: # Negative 클래스 (real)
                if pred == 0: tn += 1
                else: fp += 1

    # 지표 산출
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    avg_time = np.mean(inference_times)

    # 결과 출력
    print(f"\n========== [{model_name}] 성능 평가 결과 ==========")
    print(f"Accuracy       : {accuracy*100:05.2f}%")
    print(f"Precision      : {precision*100:05.2f}%")
    print(f"Recall         : {recall*100:05.2f}%  ← 핵심")
    print(f"F1 Score       : {f1*100:05.2f}%")
    print(f"FPR            : {fpr*100:05.2f}%")
    print(f"Inference Time : {avg_time:0.2f}초")
    print("------------------------------------")
    print(f"TP: {tp:02}  FP: {fp:02}  FN: {fn:02}  TN: {tn:02}")
    print("====================================\n")

## 3. 실행 예시
`test_data` 폴더에 `real`, `fake` 하위 폴더를 만들고 이미지를 넣은 후 실행하세요.

In [4]:
# 실제 데이터 경로로 수정 필요
TEST_PATH = './test_dataset'

if os.path.exists(TEST_PATH):
    run_benchmark(model_omniaid, "OmniAID", TEST_PATH)
else:
    print(f"경로를 찾을 수 없습니다: {TEST_PATH}. 테스트용 이미지를 준비해 주세요.")


========== [OmniAID] 성능 평가 결과 ==========
Accuracy       : 93.14%
Precision      : 97.53%
Recall         : 78.22%  ← 핵심
F1 Score       : 86.81%
FPR            : 00.80%
Inference Time : 4.00초
------------------------------------
TP: 79  FP: 02  FN: 22  TN: 247

